# EEG Dataset Unit Check & Loader Test
KL 2026-01-06

In [15]:
data_dir = "/mnt/dataset2/Processed_datasets/EEG_Bench/ChineseEEG2_reading_binary/ses-garnettdream"

In [6]:
import sys
sys.path.insert(0, '.')

from pathlib import Path
from loader import load_dataset, split_dataset, SplitStrategy
from hdf5_io import HDF5Reader
import numpy as np

In [7]:
def detect_unit(data: np.ndarray) -> tuple[str, float]:
    """Detect data unit based on amplitude range."""
    max_amp = np.abs(data).max()
    if max_amp > 1e-2:  # > 0.01, likely µV
        return "µV", max_amp
    return "V", max_amp


def check_dataset_unit(data_dir: str) -> dict:
    """Check unit and amplitude statistics for a dataset."""
    h5_files = sorted(Path(data_dir).glob("*_*.h5"))
    print(f"Found {len(h5_files)} HDF5 files in {data_dir}")
    if not h5_files:
        return {"error": "No HDF5 files found"}

    stats = {
        "dataset_name": None,
        "num_files": len(h5_files),
        "num_labels": None,
        "category_list": None,
        "detected_unit": None,
        "max_amplitude": 0,
        "min_amplitude": float('inf'),
        "mean_amplitude": [],
        "samples_checked": 0,
        "amplitude_violations": 0,
    }

    for h5_file in h5_files[:]:
        print(f"Checking file: {h5_file}")
        with HDF5Reader(str(h5_file)) as reader:
            subj = reader.subject_attrs
            if stats["dataset_name"] is None:
                stats["dataset_name"] = subj.dataset_name
                stats["num_labels"] = getattr(subj, 'num_labels', None)
                stats["category_list"] = getattr(subj, 'category_list', None)

            trial_names = reader.get_trial_names()[:3]
            for trial_name in trial_names:
                seg_names = reader.get_segment_names(trial_name)[:5]
                # `subj = reader.subject_attrs` is assigning the attribute `subject_attrs` of the `reader`
                # object to the variable `subj`. This line of code is retrieving the subject attributes from
                # the HDF5 file being read by the `reader` object. These attributes typically contain
                # information about the dataset, task type, subject ID, sampling rate, montage, channels, and
                # other relevant metadata related to the EEG data being processed.
                for seg_name in seg_names:
                    seg = reader.get_segment(trial_name, seg_name)
                    data = seg.data
                    max_amp = np.abs(data).max()
                    stats["max_amplitude"] = max(stats["max_amplitude"], max_amp)
                    stats["min_amplitude"] = min(stats["min_amplitude"], max_amp)
                    stats["mean_amplitude"].append(np.abs(data).mean())
                    stats["samples_checked"] += 1
                    if max_amp > 600:
                        stats["amplitude_violations"] += 1

    stats["detected_unit"], _ = detect_unit(np.array([stats["max_amplitude"]]))
    stats["mean_amplitude"] = np.mean(stats["mean_amplitude"]) if stats["mean_amplitude"] else 0
    return stats

## Single Dataset Report

In [16]:
stats = check_dataset_unit(data_dir)

print(f"Dataset: {stats['dataset_name']}")
print(f"Files: {stats['num_files']}")
print(f"Samples checked: {stats['samples_checked']}")
print()
print("--- Label Info ---")
print(f"Num labels: {stats['num_labels']}")
print(f"Categories: {stats['category_list']}")
print()
print("--- Unit Analysis ---")
print(f"Detected unit: {stats['detected_unit']}")
print(f"Max amplitude: {stats['max_amplitude']:.4e}")
print(f"Min amplitude: {stats['min_amplitude']:.4e}")
print(f"Mean amplitude: {stats['mean_amplitude']:.4e}")
print()
print("--- Validation ---")
is_uv = stats['detected_unit'] == 'µV'
has_labels = stats['num_labels'] is not None and stats['num_labels'] > 0
has_categories = stats['category_list'] is not None and len(stats['category_list']) > 0
print(f"Unit is µV: {'✓' if is_uv else '✗ (NEEDS UPDATE)'}")
print(f"Has num_labels: {'✓' if has_labels else '✗ (NEEDS UPDATE)'}")
print(f"Has category_list: {'✓' if has_categories else '✗ (NEEDS UPDATE)'}")
if is_uv:
    print(f"Amplitude violations (>600 µV): {stats['amplitude_violations']}/{stats['samples_checked']}")

Found 0 HDF5 files in /mnt/dataset2/Processed_datasets/EEG_Bench/ChineseEEG2_reading_binary/ses-garnettdream


KeyError: 'dataset_name'

## Batch Loading Test

In [12]:
loader = load_dataset(data_dir, batch_size=32, num_workers=0)
print(f"Total samples: {len(loader.dataset)}")

for i, batch in enumerate(loader):
    if i >= 3:
        break
    unit, max_amp = detect_unit(batch['data'].numpy())
    print(f"Batch {i}: shape={batch['data'].shape}, unit={unit}, max_amp={max_amp:.2e}, labels={batch['label'].squeeze().tolist()[:5]}...")

Total samples: 12461
Batch 0: shape=torch.Size([32, 19, 2000]), unit=V, max_amp=0.00e+00, labels=[1, 0, 0, 0, 0]...
Batch 1: shape=torch.Size([32, 19, 2000]), unit=V, max_amp=0.00e+00, labels=[0, 0, 1, 1, 0]...
Batch 2: shape=torch.Size([32, 19, 2000]), unit=V, max_amp=0.00e+00, labels=[0, 1, 0, 1, 0]...


In [14]:
# Inspect a single subject's channels and labels
from pathlib import Path
from hdf5_io import HDF5Reader


def inspect_single_subject(h5_path: str):
    """Print channel info and label distribution for a single subject HDF5 file.

    Args:
        h5_path: Full path to a single subject HDF5 file (e.g., /mnt/.../sub_1.h5)
    """
    h5_path = str(h5_path)
    p = Path(h5_path)
    if not p.exists():
        print(f"File not found: {h5_path}")
        return

    print("=" * 80)
    print(f"Inspecting subject file: {h5_path}")
    print("=" * 80)

    try:
        with HDF5Reader(h5_path) as reader:
            subj = reader.subject_attrs

            # Subject-level info
            print("1. Subject-Level Information")
            print("-" * 80)
            print(f"Dataset name: {subj.dataset_name}")
            print(f"Subject ID: {subj.subject_id}")
            print(f"Sampling rate: {subj.rsFreq} Hz")
            print(f"Montage: {subj.montage}")
            chn_name = list(subj.chn_name)
            print(f"Channels ({len(chn_name)}): {chn_name}")
            print(f"Num labels: {getattr(subj, 'num_labels', None)}")
            print(f"Categories: {getattr(subj, 'category_list', None)}")

            # Collect all labels from segments
            print("\n2. Segment-Level Label Distribution")
            print("-" * 80)
            from collections import Counter

            label_counts = Counter()
            trial_names = reader.get_trial_names()
            for trial_name in trial_names:
                seg_names = reader.get_segment_names(trial_name)
                for seg_name in seg_names:
                    seg = reader.get_segment(trial_name, seg_name)
                    if seg.segment.label is not None and len(seg.segment.label) > 0:
                        label_val = int(seg.segment.label[0])
                        label_counts[label_val] += 1

            total_segments = sum(label_counts.values())
            if total_segments == 0:
                print("No labels found in segments")
            else:
                print(f"Total segments: {total_segments}")
                print(f"{'Label':<10} {'Category':<20} {'Count':<15} {'Percentage':<10}")
                print("-" * 60)
                cat_list = getattr(subj, 'category_list', None)
                for label_val in sorted(label_counts.keys()):
                    count = label_counts[label_val]
                    pct = count / total_segments * 100
                    if cat_list and label_val < len(cat_list):
                        cat_name = cat_list[label_val]
                    else:
                        cat_name = f"Label_{label_val}"
                    print(f"{label_val:<10} {cat_name:<20} {count:<15} {pct:>6.2f}%")

            print("\n3. Trial and Segment Structure (summary)")
            print("-" * 80)
            print(f"Number of trials: {len(trial_names)}")
            if trial_names:
                first_trial = trial_names[0]
                seg_names = reader.get_segment_names(first_trial)
                print(f"First trial: {first_trial}, segments: {len(seg_names)}")
                if seg_names:
                    seg = reader.get_segment(first_trial, seg_names[0])
                    print(f"Example segment shape: {seg.data.shape}")
                    print(f"Example segment label: {seg.segment.label}")

            print("\n" + "=" * 80)

    except Exception as e:
        print(f"Error inspecting {h5_path}: {e}")
        import traceback
        traceback.print_exc()


# Example usage (edit the path below to inspect a specific subject)
inspect_single_subject("/mnt/dataset2/Processed_datasets/EEG_Bench/TUEP_02_tcp_le/TUEP/sub_aaaaaanr.h5")


Inspecting subject file: /mnt/dataset2/Processed_datasets/EEG_Bench/TUEP_02_tcp_le/TUEP/sub_aaaaaanr.h5
1. Subject-Level Information
--------------------------------------------------------------------------------
Dataset name: TUEP_unknown
Subject ID: aaaaaanr
Sampling rate: 200.0 Hz
Montage: 10_20
Channels (19): ['FP1', 'FP2', 'F7', 'F3', 'FZ', 'F4', 'F8', 'T7', 'C3', 'CZ', 'C4', 'T8', 'P7', 'P3', 'PZ', 'P4', 'P8', 'O1', 'O2']
Num labels: 2
Categories: ['non_seizure', 'seizure']

2. Segment-Level Label Distribution
--------------------------------------------------------------------------------
Total segments: 259
Label      Category             Count           Percentage
------------------------------------------------------------
1          seizure              259             100.00%

3. Trial and Segment Structure (summary)
--------------------------------------------------------------------------------
Number of trials: 3
First trial: trial0, segments: 121
Example segment shape:

## HDF5 Metadata Inspection

In [48]:
h5_files = sorted(Path(data_dir).glob("sub_*.h5"))
if h5_files:
    with HDF5Reader(str(h5_files[0])) as reader:
        subj = reader.subject_attrs
        print(f"Dataset name: {subj.dataset_name}")
        print(f"Task type: {subj.task_type}")
        print(f"Downstream task: {subj.downstream_task_type}")
        print(f"Subject ID: {subj.subject_id}")
        print(f"Sampling rate: {subj.rsFreq} Hz")
        print(f"Montage: {subj.montage}")
        print(f"Channels ({len(subj.chn_name)}): {subj.chn_name[:]}")
        print(f"Num labels: {getattr(subj, 'num_labels', 'N/A')}")
        print(f"Categories: {getattr(subj, 'category_list', 'N/A')}")
        print(f"Num trials: {len(reader.get_trial_names())}")
        print(f"Num segments: {len(reader)}")

Dataset name: TUEP_eval
Task type: seizure
Downstream task: classification
Subject ID: aaaaaaaq
Sampling rate: 200.0 Hz
Montage: 10_20
Channels (21): ['FP1', 'FP2', 'F7', 'F3', 'FZ', 'F4', 'F8', 'T7', 'C3', 'CZ', 'C4', 'T8', 'P7', 'P3', 'PZ', 'P4', 'P8', 'O1', 'O2', 'A1', 'A2']
Num labels: 2
Categories: ['non_seizure', 'seizure']
Num trials: 10
Num segments: 912


In [49]:
def analyze_all_subjects_channels(data_dir: str):
    """Analyze channel information across all subjects."""
    h5_files = sorted(Path(data_dir).glob("sub_*.h5"))
    
    if not h5_files:
        print("No HDF5 files found")
        return
    
    print("=" * 80)
    print("Channel Statistics Across All Subjects")
    print("=" * 80)
    
    # Collect channel information from all subjects
    all_channel_lists = []
    channel_sets = []
    subject_channel_info = []
    
    for h5_file in h5_files:
        try:
            with HDF5Reader(str(h5_file)) as reader:
                subj = reader.subject_attrs
                chn_name = subj.chn_name if subj.chn_name else []
                
                if chn_name:
                    all_channel_lists.append(chn_name)
                    channel_sets.append(set(chn_name))
                    subject_channel_info.append({
                        'subject_id': subj.subject_id,
                        'file': h5_file.name,
                        'channels': chn_name,
                        'num_channels': len(chn_name),
                        'channel_set': set(chn_name),
                    })
        except Exception as e:
            print(f"Error reading {h5_file.name}: {e}")
            continue
    
    if not subject_channel_info:
        print("No valid subjects found")
        return
    
    # 1. Channel count statistics
    print(f"\n1. Channel Count Statistics")
    print("-" * 80)
    channel_counts = [info['num_channels'] for info in subject_channel_info]
    unique_counts = sorted(set(channel_counts))
    count_distribution = {count: channel_counts.count(count) for count in unique_counts}
    
    print(f"Total subjects: {len(subject_channel_info)}")
    print(f"Unique channel counts: {unique_counts}")
    print(f"\nChannel count distribution:")
    for count, num_subjects in sorted(count_distribution.items()):
        pct = num_subjects / len(subject_channel_info) * 100
        print(f"  {count} channels: {num_subjects} subjects ({pct:.1f}%)")
    
    # 2. Channel consistency
    print(f"\n2. Channel Consistency Analysis")
    print("-" * 80)
    
    common_channels = set()
    all_unique_channels = set()
    if channel_sets:
        common_channels = set.intersection(*channel_sets)
        all_unique_channels = set.union(*channel_sets)
        
        print(f"Common channels (present in ALL subjects): {len(common_channels)}")
        print(f"All unique channels (across all subjects): {len(all_unique_channels)}")
        
        if len(common_channels) == len(all_unique_channels):
            print("✓ All subjects have exactly the same channels")
        else:
            print(f"⚠ Channel differences detected:")
            missing_channels = all_unique_channels - common_channels
            print(f"  Channels missing in some subjects: {len(missing_channels)}")
            
            # Show which channels are missing in which subjects
            print(f"\n  Missing channel details:")
            for ch in sorted(missing_channels):
                subjects_with_ch = [info['subject_id'] for info in subject_channel_info if ch in info['channel_set']]
                subjects_without_ch = [info['subject_id'] for info in subject_channel_info if ch not in info['channel_set']]
                print(f"    {ch}:")
                print(f"      Present in: {len(subjects_with_ch)} subjects ({', '.join(map(str, subjects_with_ch[:10]))}{'...' if len(subjects_with_ch) > 10 else ''})")
                print(f"      Missing in: {len(subjects_without_ch)} subjects ({', '.join(map(str, subjects_without_ch[:10]))}{'...' if len(subjects_without_ch) > 10 else ''})")
    
    # 3. Channel order consistency
    print(f"\n3. Channel Order Consistency")
    print("-" * 80)
    
    order_consistent = True
    if all_channel_lists:
        first_channels = all_channel_lists[0]
        order_consistent = all(ch == first_channels for ch in all_channel_lists)
        
        if order_consistent:
            print("✓ All subjects have the same channel order")
        else:
            print("⚠ Channel order differs across subjects")
            inconsistent_subjects = []
            for i, (info, ch_list) in enumerate(zip(subject_channel_info, all_channel_lists)):
                if ch_list != first_channels:
                    inconsistent_subjects.append(info['subject_id'])
            print(f"  Subjects with different order: {len(inconsistent_subjects)}")
            if inconsistent_subjects:
                print(f"  Subject IDs: {', '.join(map(str, inconsistent_subjects[:10]))}{'...' if len(inconsistent_subjects) > 10 else ''}")
    
    # 4. Channel frequency (how many subjects have each channel)
    print(f"\n4. Channel Frequency Analysis")
    print("-" * 80)
    
    channel_frequency = {}
    if all_unique_channels:
        for ch in all_unique_channels:
            count = sum(1 for info in subject_channel_info if ch in info['channel_set'])
            channel_frequency[ch] = count
        
        # Sort by frequency
        sorted_channels = sorted(channel_frequency.items(), key=lambda x: x[1], reverse=True)
        
        print(f"Channel frequency (how many subjects have each channel):")
        print(f"{'Channel':<15} {'Count':<10} {'Percentage':<10}")
        print("-" * 40)
        for ch, count in sorted_channels:
            pct = count / len(subject_channel_info) * 100
            marker = "✓" if count == len(subject_channel_info) else "⚠"
            print(f"{ch:<15} {count:<10} {pct:>6.1f}%  {marker}")
    
    # 5. Subject-by-subject channel summary
    print(f"\n5. Subject-by-Subject Channel Summary")
    print("-" * 80)
    print(f"{'Subject ID':<20} {'Channels':<10} {'Channel List':<50}")
    print("-" * 80)
    
    for info in subject_channel_info[:20]:  # Show first 20 subjects
        ch_list_str = ', '.join(info['channels'][:5])
        if len(info['channels']) > 5:
            ch_list_str += '...'
        print(f"{str(info['subject_id']):<20} {info['num_channels']:<10} {ch_list_str:<50}")
    
    if len(subject_channel_info) > 20:
        print(f"... and {len(subject_channel_info) - 20} more subjects")
    
    # 6. Summary
    print(f"\n6. Summary")
    print("-" * 80)
    print(f"Total subjects analyzed: {len(subject_channel_info)}")
    print(f"Common channels: {len(common_channels)}")
    print(f"All unique channels: {len(all_unique_channels)}")
    print(f"Channel count consistency: {'✓' if len(unique_counts) == 1 else '⚠'}")
    print(f"Channel set consistency: {'✓' if len(common_channels) == len(all_unique_channels) else '⚠'}")
    print(f"Channel order consistency: {'✓' if order_consistent else '⚠'}")
    
    print("\n" + "=" * 80)
    
    return {
        'subject_channel_info': subject_channel_info,
        'common_channels': sorted(list(common_channels)) if common_channels else [],
        'all_unique_channels': sorted(list(all_unique_channels)) if all_unique_channels else [],
        'channel_frequency': channel_frequency,
    }


# Run channel statistics analysis
channel_stats = analyze_all_subjects_channels(data_dir)

Channel Statistics Across All Subjects

1. Channel Count Statistics
--------------------------------------------------------------------------------
Total subjects: 71
Unique channel counts: [21]

Channel count distribution:
  21 channels: 71 subjects (100.0%)

2. Channel Consistency Analysis
--------------------------------------------------------------------------------
Common channels (present in ALL subjects): 21
All unique channels (across all subjects): 21
✓ All subjects have exactly the same channels

3. Channel Order Consistency
--------------------------------------------------------------------------------
✓ All subjects have the same channel order

4. Channel Frequency Analysis
--------------------------------------------------------------------------------
Channel frequency (how many subjects have each channel):
Channel         Count      Percentage
----------------------------------------
C3              71          100.0%  ✓
P3              71          100.0%  ✓
O1       

# Inspect a single subject's channels and labels

In [50]:

from pathlib import Path
from hdf5_io import HDF5Reader
import numpy as np


def inspect_single_subject(h5_path: str):
    """Print channel info and label distribution for a single subject HDF5 file.

    Args:
        h5_path: Full path to a single subject HDF5 file (e.g., /mnt/.../sub_1.h5)
    """
    h5_path = str(h5_path)
    p = Path(h5_path)
    if not p.exists():
        print(f"File not found: {h5_path}")
        return

    print("=" * 80)
    print(f"Inspecting subject file: {h5_path}")
    print("=" * 80)

    try:
        with HDF5Reader(h5_path) as reader:
            subj = reader.subject_attrs

            # 1. Subject-level information
            print("1. Subject-Level Information")
            print("-" * 80)
            print(f"Dataset name: {subj.dataset_name}")
            print(f"Subject ID: {subj.subject_id}")
            print(f"Sampling rate: {subj.rsFreq} Hz")
            print(f"Montage: {subj.montage}")
            chn_name = list(subj.chn_name)
            print(f"Channels ({len(chn_name)}): {chn_name}")
            print(f"Num labels: {getattr(subj, 'num_labels', None)}")
            print(f"Categories: {getattr(subj, 'category_list', None)}")

            # 2. Segment-level label distribution
            print("\n2. Segment-Level Label Distribution")
            print("-" * 80)
            from collections import Counter

            label_counts = Counter()
            trial_names = reader.get_trial_names()
            for trial_name in trial_names:
                seg_names = reader.get_segment_names(trial_name)
                for seg_name in seg_names:
                    seg = reader.get_segment(trial_name, seg_name)
                    if seg.segment.label is not None and len(seg.segment.label) > 0:
                        label_val = int(seg.segment.label[0])
                        label_counts[label_val] += 1

            total_segments = sum(label_counts.values())
            if total_segments == 0:
                print("No labels found in segments")
            else:
                print(f"Total segments: {total_segments}")
                print(f"{'Label':<10} {'Category':<20} {'Count':<15} {'Percentage':<10}")
                print("-" * 60)
                cat_list = getattr(subj, 'category_list', None)
                for label_val in sorted(label_counts.keys()):
                    count = label_counts[label_val]
                    pct = count / total_segments * 100
                    if cat_list and label_val < len(cat_list):
                        cat_name = cat_list[label_val]
                    else:
                        cat_name = f"Label_{label_val}"
                    print(f"{label_val:<10} {cat_name:<20} {count:<15} {pct:>6.2f}%")

            # 3. Trial and segment structure summary
            print("\n3. Trial and Segment Structure (summary)")
            print("-" * 80)
            print(f"Number of trials: {len(trial_names)}")
            if trial_names:
                first_trial = trial_names[0]
                seg_names = reader.get_segment_names(first_trial)
                print(f"First trial: {first_trial}, segments: {len(seg_names)}")
                if seg_names:
                    seg = reader.get_segment(first_trial, seg_names[0])
                    print(f"Example segment shape: {seg.data.shape}")
                    print(f"Example segment label: {seg.segment.label}")

            print("\n" + "=" * 80)

    except Exception as e:
        print(f"Error inspecting {h5_path}: {e}")
        import traceback
        traceback.print_exc()


# Example usage (edit the path below to inspect a specific subject)
# inspect_single_subject("/mnt/dataset2/Processed_datasets/EEG_Bench/TUEP/sub_aaaaaaaq.h5")

In [ ]:
inspect_single_subject(data_dir+"/sub_aaaaaaaq.h5")

Inspecting subject file: /mnt/dataset2/Processed_datasets/EEG_Bench/TUEP/sub_aaaaaaaq.h5
1. Subject-Level Information
--------------------------------------------------------------------------------
Dataset name: TUEP_eval
Subject ID: aaaaaaaq
Sampling rate: 200.0 Hz
Montage: 10_20
Channels (21): ['FP1', 'FP2', 'F7', 'F3', 'FZ', 'F4', 'F8', 'T7', 'C3', 'CZ', 'C4', 'T8', 'P7', 'P3', 'PZ', 'P4', 'P8', 'O1', 'O2', 'A1', 'A2']
Num labels: 2
Categories: ['non_seizure', 'seizure']

2. Segment-Level Label Distribution
--------------------------------------------------------------------------------


## Label Statistics Across All Subjects

In [42]:
def analyze_all_subjects_labels(data_dir: str):
    """Analyze label information across all subjects."""
    h5_files = sorted(Path(data_dir).glob("sub_*.h5"))
    
    if not h5_files:
        print("No HDF5 files found")
        return
    
    print("=" * 80)
    print("Label Statistics Across All Subjects")
    print("=" * 80)
    
    # Collect label information from all subjects
    subject_label_info = []
    all_labels = []
    label_distribution = {}
    category_lists = []
    
    for h5_file in h5_files:
        try:
            with HDF5Reader(str(h5_file)) as reader:
                subj = reader.subject_attrs
                
                # Get category list
                category_list = getattr(subj, 'category_list', None)
                if category_list:
                    category_lists.append(category_list)
                
                # Collect labels from all segments
                trial_names = reader.get_trial_names()
                subject_labels = []
                label_counts = {}
                
                for trial_name in trial_names:
                    seg_names = reader.get_segment_names(trial_name)
                    for seg_name in seg_names:
                        seg = reader.get_segment(trial_name, seg_name)
                        label_val = int(seg.segment.label[0]) if len(seg.segment.label) > 0 else None
                        if label_val is not None:
                            subject_labels.append(label_val)
                            all_labels.append(label_val)
                            label_counts[label_val] = label_counts.get(label_val, 0) + 1
                
                subject_label_info.append({
                    'subject_id': subj.subject_id,
                    'file': h5_file.name,
                    'num_labels': getattr(subj, 'num_labels', None),
                    'category_list': category_list,
                    'label_counts': label_counts,
                    'total_segments': len(subject_labels),
                })
        except Exception as e:
            print(f"Error reading {h5_file.name}: {e}")
            import traceback
            traceback.print_exc()
            continue
    
    if not subject_label_info:
        print("No valid subjects found")
        return
    
    # 1. Dataset-level label information
    print(f"\n1. Dataset-Level Label Information")
    print("-" * 80)
    
    # Get category list from first subject (assuming consistent)
    first_category_list = subject_label_info[0]['category_list']
    num_labels = subject_label_info[0]['num_labels']
    
    print(f"Number of labels: {num_labels}")
    print(f"Category list: {first_category_list}")
    print(f"Total subjects: {len(subject_label_info)}")
    print(f"Total segments: {len(all_labels)}")
    
    # 2. Overall label distribution
    print(f"\n2. Overall Label Distribution")
    print("-" * 80)
    
    from collections import Counter
    overall_label_counts = Counter(all_labels)
    
    print(f"{'Label':<10} {'Category':<20} {'Count':<15} {'Percentage':<10}")
    print("-" * 60)
    for label_val in sorted(overall_label_counts.keys()):
        count = overall_label_counts[label_val]
        pct = count / len(all_labels) * 100
        category_name = first_category_list[label_val] if first_category_list and label_val < len(first_category_list) else f"Label_{label_val}"
        print(f"{label_val:<10} {category_name:<20} {count:<15} {pct:>6.2f}%")
    
    # 3. Label consistency across subjects
    print(f"\n3. Label Consistency Across Subjects")
    print("-" * 80)
    
    # Check if all subjects have the same category_list
    category_list_consistent = all(
        info['category_list'] == first_category_list 
        for info in subject_label_info 
        if info['category_list']
    )
    
    # Check if all subjects have the same num_labels
    num_labels_list = [info['num_labels'] for info in subject_label_info if info['num_labels'] is not None]
    num_labels_consistent = len(set(num_labels_list)) == 1 if num_labels_list else False
    
    print(f"Category list consistency: {'✓' if category_list_consistent else '⚠'}")
    print(f"Number of labels consistency: {'✓' if num_labels_consistent else '⚠'}")
    
    if not category_list_consistent:
        print("\n  Category list differences:")
        unique_category_lists = {}
        for info in subject_label_info:
            cat_key = tuple(info['category_list']) if info['category_list'] else None
            if cat_key not in unique_category_lists:
                unique_category_lists[cat_key] = []
            unique_category_lists[cat_key].append(info['subject_id'])
        
        for cat_list, subjects in unique_category_lists.items():
            print(f"    {list(cat_list) if cat_list else 'None'}: {len(subjects)} subjects")
            print(f"      Subjects: {', '.join(map(str, subjects[:10]))}{'...' if len(subjects) > 10 else ''}")
    
    # 4. Label distribution per subject
    print(f"\n4. Label Distribution Per Subject")
    print("-" * 80)
    
    # Check which labels appear in each subject
    label_presence = {}
    for label_val in sorted(overall_label_counts.keys()):
        subjects_with_label = [
            info['subject_id'] 
            for info in subject_label_info 
            if label_val in info['label_counts']
        ]
        label_presence[label_val] = subjects_with_label
    
    print(f"Label presence across subjects:")
    for label_val in sorted(label_presence.keys()):
        subjects = label_presence[label_val]
        category_name = first_category_list[label_val] if first_category_list and label_val < len(first_category_list) else f"Label_{label_val}"
        pct = len(subjects) / len(subject_label_info) * 100
        marker = "✓" if len(subjects) == len(subject_label_info) else "⚠"
        print(f"  {label_val} ({category_name}): {len(subjects)}/{len(subject_label_info)} subjects ({pct:.1f}%) {marker}")
        if len(subjects) < len(subject_label_info):
            missing_subjects = [s for s in [info['subject_id'] for info in subject_label_info] if s not in subjects]
            print(f"    Missing in: {', '.join(map(str, missing_subjects[:10]))}{'...' if len(missing_subjects) > 10 else ''}")
    
    # 5. Subject-by-subject label summary
    print(f"\n5. Subject-by-Subject Label Summary")
    print("-" * 80)
    print(f"{'Subject ID':<20} {'Segments':<12} {'Label Distribution':<50}")
    print("-" * 80)
    
    for info in subject_label_info[:20]:  # Show first 20 subjects
        label_dist_str = ', '.join([f"{k}:{v}" for k, v in sorted(info['label_counts'].items())])
        if len(label_dist_str) > 50:
            label_dist_str = label_dist_str[:47] + '...'
        print(f"{str(info['subject_id']):<20} {info['total_segments']:<12} {label_dist_str:<50}")
    
    if len(subject_label_info) > 20:
        print(f"... and {len(subject_label_info) - 20} more subjects")
    
    # 6. Label balance analysis
    print(f"\n6. Label Balance Analysis")
    print("-" * 80)
    
    # Calculate per-subject label balance
    balance_ratios = []
    for info in subject_label_info:
        if info['total_segments'] > 0 and info['label_counts']:
            counts = list(info['label_counts'].values())
            if len(counts) > 1:
                min_count = min(counts)
                max_count = max(counts)
                balance_ratio = min_count / max_count if max_count > 0 else 0
                balance_ratios.append(balance_ratio)
    
    if balance_ratios:
        avg_balance = np.mean(balance_ratios)
        min_balance = min(balance_ratios)
        max_balance = max(balance_ratios)
        
        print(f"Average label balance ratio (min/max): {avg_balance:.3f}")
        print(f"Min balance ratio: {min_balance:.3f}")
        print(f"Max balance ratio: {max_balance:.3f}")
        
        if avg_balance > 0.8:
            print("✓ Labels are well balanced across subjects")
        elif avg_balance > 0.5:
            print("⚠ Labels are moderately balanced")
        else:
            print("⚠ Labels are imbalanced - consider class weighting")
    
    # Overall label balance
    if len(overall_label_counts) > 1:
        overall_counts = list(overall_label_counts.values())
        overall_min = min(overall_counts)
        overall_max = max(overall_counts)
        overall_balance = overall_min / overall_max if overall_max > 0 else 0
        print(f"\nOverall dataset label balance: {overall_balance:.3f}")
        if overall_balance > 0.8:
            print("✓ Overall labels are well balanced")
        else:
            print("⚠ Overall labels are imbalanced")
    
    # 7. Summary
    print(f"\n7. Summary")
    print("-" * 80)
    print(f"Total subjects: {len(subject_label_info)}")
    print(f"Total segments: {len(all_labels)}")
    print(f"Number of unique labels: {len(overall_label_counts)}")
    print(f"Category list consistency: {'✓' if category_list_consistent else '⚠'}")
    print(f"Number of labels consistency: {'✓' if num_labels_consistent else '⚠'}")
    print(f"Average segments per subject: {len(all_labels) / len(subject_label_info):.1f}")
    
    print("\n" + "=" * 80)
    
    return {
        'subject_label_info': subject_label_info,
        'overall_label_counts': dict(overall_label_counts),
        'label_presence': label_presence,
        'category_list': first_category_list,
        'num_labels': num_labels,
    }


# Run label statistics analysis
label_stats = analyze_all_subjects_labels(data_dir)

Label Statistics Across All Subjects


KeyboardInterrupt: 

In [ ]:
# Visualize label distribution
try:
    import matplotlib.pyplot as plt
    
    if label_stats and 'overall_label_counts' in label_stats:
        # Bar plot of label distribution
        fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))
        
        # Overall label distribution
        labels = sorted(label_stats['overall_label_counts'].keys())
        counts = [label_stats['overall_label_counts'][l] for l in labels]
        category_names = [
            label_stats['category_list'][l] if label_stats['category_list'] and l < len(label_stats['category_list']) 
            else f"Label_{l}" 
            for l in labels
        ]
        
        colors = plt.cm.Set3(np.linspace(0, 1, len(labels)))
        bars = ax1.bar(range(len(labels)), counts, color=colors)
        ax1.set_xlabel('Label', fontsize=12)
        ax1.set_ylabel('Count', fontsize=12)
        ax1.set_title('Overall Label Distribution', fontsize=14, fontweight='bold')
        ax1.set_xticks(range(len(labels)))
        ax1.set_xticklabels([f"{l}\n({cat})" for l, cat in zip(labels, category_names)], fontsize=9)
        ax1.grid(True, alpha=0.3, axis='y')
        
        # Add count labels on bars
        for bar, count in zip(bars, counts):
            height = bar.get_height()
            ax1.text(bar.get_x() + bar.get_width()/2., height,
                    f'{count}\n({count/len(all_labels)*100:.1f}%)',
                    ha='center', va='bottom', fontsize=8)
        
        # Per-subject label distribution (heatmap style)
        if label_stats['subject_label_info']:
            subject_ids = [info['subject_id'] for info in label_stats['subject_label_info']]
            label_matrix = []
            for info in label_stats['subject_label_info']:
                row = [info['label_counts'].get(l, 0) for l in labels]
                label_matrix.append(row)
            
            im = ax2.imshow(label_matrix, aspect='auto', cmap='YlOrRd', interpolation='nearest')
            ax2.set_xlabel('Label', fontsize=12)
            ax2.set_ylabel('Subject', fontsize=12)
            ax2.set_title('Label Distribution Per Subject', fontsize=14, fontweight='bold')
            ax2.set_xticks(range(len(labels)))
            ax2.set_xticklabels([f"{l}\n({cat[:5]})" for l, cat in zip(labels, category_names)], fontsize=8)
            ax2.set_yticks(range(len(subject_ids)))
            ax2.set_yticklabels([str(sid) for sid in subject_ids], fontsize=7)
            
            # Add colorbar
            plt.colorbar(im, ax=ax2, label='Segment Count')
        
        plt.tight_layout()
        plt.show()
        
        # Pie chart of overall distribution
        fig, ax = plt.subplots(figsize=(8, 8))
        wedges, texts, autotexts = ax.pie(counts, labels=category_names, autopct='%1.1f%%', 
                                          startangle=90, colors=colors)
        ax.set_title('Label Distribution (Pie Chart)', fontsize=14, fontweight='bold')
        
        # Improve text appearance
        for autotext in autotexts:
            autotext.set_color('white')
            autotext.set_fontweight('bold')
        
        plt.tight_layout()
        plt.show()
        
    else:
        print("No label statistics available for visualization")
        
except ImportError:
    print("Matplotlib not available, skipping visualization")
except Exception as e:
    print(f"Error creating visualization: {e}")
    import traceback
    traceback.print_exc()